[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/davis-mironga/kitui-washlab-analysis/blob/main/notebooks/07_Report_Figures.ipynb)


# Notebook 07 — Report Figures
**Project:** WASHLAB Climate-Smart WASH Pilot — Kitui County  
**Analyst:** Davis Mironga  
**Purpose:** Produce all publication-ready figures and standalone deliverable maps for the Phase 1 report.

**Requires:** Outputs from Notebooks 01 to 03 in `Kitui_WASHLAB/outputs/` and GEE Assets from Notebook 01.

---
## Phase 1 deliverables produced here

| Figure | Deliverable | Source |
|--------|-------------|--------|
| Fig 1 | Seasonal water availability and water source loss map | JRC Global Surface Water via GEE |
| Fig 2 | Vegetation stress and land condition map | GEE Assets: kitui_ndvi_mean + baseline |
| Fig 3 | WASI ward choropleth | Notebook 02 output |
| Fig 4 | WASI component breakdown chart | Notebook 02 output |
| Fig 5 | Hotspot analysis map | Notebook 03 output |

## Phase 2 figures (added once borehole data received)
- Fig 6: Borehole network overview
- Fig 7: Coverage gap map

## Outputs
All figures saved to `Kitui_WASHLAB/outputs/maps/` at 300 DPI.
Report tables saved to `Kitui_WASHLAB/outputs/report/`.


### 1. Setup

Installs required libraries, mounts Google Drive, authenticates GEE, and defines constants and helper functions.
The `load_asset` function downloads rasters from GEE Assets with automatic scale capping to stay under the 50MB request limit.
The `download_jrc_band` function downloads JRC water layers directly from the GEE source collection.
The county boundary mask is defined in the next cell after ward boundaries are loaded.

Run this cell first before anything else.


In [ ]:
# ── Setup ─────────────────────────────────────────────────────────────────────
!pip install geopandas matplotlib rasterio earthengine-api geemap requests rasterstats -q

import numpy as np
import pandas as pd
import geopandas as gpd
import rasterio
from rasterio.io import MemoryFile
from rasterio.warp import reproject
from rasterio.enums import Resampling
from rasterio.features import geometry_mask
from rasterio.transform import from_bounds
import rasterstats
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.colors as mcolors
from shapely.geometry import mapping, Point
import requests
import os
import warnings
warnings.filterwarnings('ignore')

import ee
import geemap
from google.colab import drive

drive.mount('/content/drive')
DRIVE = '/content/drive/MyDrive/Kitui_WASHLAB/'
OUT   = DRIVE + 'outputs/'
MAPS  = OUT + 'maps/'
REPT  = OUT + 'report/'
os.makedirs(MAPS, exist_ok=True)
os.makedirs(REPT, exist_ok=True)

# GEE authentication
GEE_PROJECT  = 'kitui-washlab-analysis'
ASSET_FOLDER = f'projects/{GEE_PROJECT}/assets/kitui'
ee.Authenticate()
ee.Initialize(project=GEE_PROJECT)

WGS84  = 'EPSG:4326'
BOUNDS = {'west': 37.5, 'east': 39.2, 'south': -3.1, 'north': 0.0}

# Output grid — matches Notebook 02
PIXEL_DEG  = 500 / 111320
GRID_W     = int((BOUNDS['east'] - BOUNDS['west']) / PIXEL_DEG)
GRID_H     = int((BOUNDS['north'] - BOUNDS['south']) / PIXEL_DEG)
GRID_TRANS = from_bounds(
    BOUNDS['west'], BOUNDS['south'], BOUNDS['east'], BOUNDS['north'],
    GRID_W, GRID_H
)
GRID_SHAPE = (GRID_H, GRID_W)

# Consistent figure style
plt.rcParams.update({
    'font.family':    'DejaVu Sans',
    'font.size':      10,
    'axes.titlesize': 11,
    'axes.labelsize': 9,
    'savefig.dpi':    300,
    'savefig.bbox':   'tight',
})

def add_north_arrow(ax, x=0.95, y=0.10):
    ax.annotate('N', xy=(x, y), xytext=(x, y - 0.05),
                xycoords='axes fraction', fontsize=12,
                ha='center', fontweight='bold',
                arrowprops=dict(arrowstyle='->', color='black', lw=1.5))

def add_caption(ax, text):
    ax.text(0.5, -0.06, text, transform=ax.transAxes,
            ha='center', va='top', fontsize=7.5,
            style='italic', color='#555555')

# Scale caps to stay within GEE 50MB per-request limit
SCALE_CAPS = {30: 150, 100: 250, 500: 500, 1000: 1000, 5000: 5000, 9000: 9000}

def load_asset(asset_name, download_scale):
    """
    Download a GEE asset and reproject to the common 500m output grid.
    Scale is capped automatically to stay under the GEE 50MB limit.
    Returns a raw 2D float32 array — apply mask() separately after loading.
    """
    safe_scale = SCALE_CAPS.get(download_scale, download_scale)
    if safe_scale != download_scale:
        print(f'  Scale capped: {download_scale}m -> {safe_scale}m (50MB limit)')
    url = ee.Image(f'{ASSET_FOLDER}/{asset_name}').getDownloadURL({
        'scale': safe_scale, 'crs': WGS84,
        'region': ee.Geometry.BBox(
            BOUNDS['west'], BOUNDS['south'],
            BOUNDS['east'], BOUNDS['north']
        ),
        'format': 'GEO_TIFF',
    })
    resp = requests.get(url, timeout=300)
    resp.raise_for_status()
    out = np.full(GRID_SHAPE, np.nan, dtype=np.float32)
    with MemoryFile(resp.content) as mem:
        with mem.open() as src:
            reproject(
                source=rasterio.band(src, 1), destination=out,
                src_transform=src.transform, src_crs=src.crs,
                dst_transform=GRID_TRANS, dst_crs=WGS84,
                resampling=Resampling.bilinear,
                src_nodata=src.nodata, dst_nodata=np.nan
            )
    return out

def download_jrc_band(band_name, scale=150):
    """
    Download a JRC Global Surface Water band directly from the GEE source collection.
    Uses nearest-neighbour resampling to preserve discrete class values.
    """
    kitui_bbox = ee.Geometry.BBox(
        BOUNDS['west'], BOUNDS['south'],
        BOUNDS['east'], BOUNDS['north']
    )
    image = ee.Image('JRC/GSW1_4/GlobalSurfaceWater').clip(kitui_bbox).select(band_name)
    url = image.getDownloadURL({
        'scale': scale, 'crs': WGS84,
        'region': kitui_bbox,
        'format': 'GEO_TIFF',
    })
    resp = requests.get(url, timeout=300)
    resp.raise_for_status()
    out = np.full(GRID_SHAPE, np.nan, dtype=np.float32)
    with MemoryFile(resp.content) as mem:
        with mem.open() as src:
            reproject(
                source=rasterio.band(src, 1), destination=out,
                src_transform=src.transform, src_crs=src.crs,
                dst_transform=GRID_TRANS, dst_crs=WGS84,
                resampling=Resampling.nearest,
                src_nodata=src.nodata, dst_nodata=np.nan
            )
    return out

print('Setup complete')
print(f'Grid: {GRID_W} x {GRID_H} pixels at ~500m')
print(f'Maps will be saved to: {MAPS}')


### 2. Load Analysis Outputs

Loads ward boundaries, WASI results, and hotspot results from Drive.
Builds the county boundary mask and polygon after wards are loaded.
The mask is applied to rasters for clean visualisation.
The polygon is used to clip scatter points strictly to the county boundary.
Phase 2 outputs are loaded if available and skipped if not.


In [ ]:
# ── Load analysis outputs ──────────────────────────────────────────────────────

# Ward boundaries
wards = gpd.read_file(DRIVE + 'boundaries/kitui_wards.shp').to_crs(WGS84)
if 'Ward' not in wards.columns:
    for col in ['NAME_3', 'WARD', 'ward', 'NAME']:
        if col in wards.columns:
            wards = wards.rename(columns={col: 'Ward'})
            break

# County geometry and mask — built from ward boundaries
county_geom = wards.dissolve().geometry.iloc[0]
county_mask = geometry_mask(
    [mapping(county_geom)],
    transform=GRID_TRANS, invert=True, out_shape=GRID_SHAPE
)

def mask(arr):
    """Apply county boundary mask to a raster array."""
    return np.where(county_mask, arr, np.nan)

def filter_to_county(lons, lats):
    """Remove scatter points that fall outside the county boundary polygon."""
    lons, lats = list(lons), list(lats)
    keep_lons, keep_lats = [], []
    for lo, la in zip(lons, lats):
        if county_geom.contains(Point(lo, la)):
            keep_lons.append(lo)
            keep_lats.append(la)
    return keep_lons, keep_lats

# WASI outputs (Notebook 02)
wasi_table = pd.read_csv(OUT + 'kitui_wasi_ward_table.csv')
wasi_wards = gpd.read_file(OUT + 'kitui_wasi_ward.geojson')

# Hotspot outputs (Notebook 03)
hotspot_wards = gpd.read_file(OUT + 'kitui_hotspot_ward.geojson')

# Phase 2 outputs — load if available
has_coverage  = os.path.exists(OUT + 'kitui_coverage_ward_table.csv')
has_boreholes = os.path.exists(OUT + 'kitui_boreholes.geojson')
if has_coverage:
    coverage_table = pd.read_csv(OUT + 'kitui_coverage_ward_table.csv')
    coverage_wards = gpd.read_file(OUT + 'kitui_coverage_ward.geojson')
if has_boreholes:
    boreholes = gpd.read_file(OUT + 'kitui_boreholes.geojson')

print(f'Wards loaded: {len(wards)}')
print(f'County mask built: {GRID_H} x {GRID_W} pixels')
print(f'WASI wards: {len(wasi_wards)}')
print(f'Hotspot wards: {len(hotspot_wards)}')
print(f'Phase 2 coverage gap: {"available" if has_coverage else "not yet available"}')
print(f'Phase 2 boreholes:    {"available" if has_boreholes else "not yet available"}')


### 3. Figure 1 — Seasonal Water Availability and Water Source Loss

**Phase 1 deliverable:** Shows the current state of surface water sources and
how much water has been lost since 1984 across Kitui County.

Data source: JRC Global Surface Water (1984 to 2021), downloaded directly from GEE.

Two panels:
- **Left** — individual water source locations: blue = still present, red = lost since 1984
- **Right** — ward-level map showing what percentage of each ward's historical water sources have been lost

JRC transition codes used:
- Code 1 = Permanent water (present)
- Code 4 = Seasonal water (present)
- Code 3 = Lost permanent water (was permanent, now gone)
- Code 6 = Lost seasonal water (was seasonal, now gone)

All scatter points are clipped strictly to the Kitui County boundary.


In [ ]:
# ── Figure 1: Seasonal water availability and water source loss ───────────────
print('Loading JRC water transition layer from GEE source collection...')
water_transition = download_jrc_band('transition', scale=150)
print(f'Transition values present: {np.unique(water_transition[~np.isnan(water_transition)])}')

# Compute ward-level water loss using zonal stats
tmp_path = '/tmp/jrc_transition.tif'
with rasterio.open(
    tmp_path, 'w', driver='GTiff',
    height=GRID_H, width=GRID_W,
    count=1, dtype='float32',
    crs='EPSG:4326', transform=GRID_TRANS,
    nodata=float('nan')
) as dst:
    dst.write(water_transition.astype(np.float32), 1)

def count_class(raster_path, gdf, value):
    stats = rasterstats.zonal_stats(
        gdf, raster_path, stats=['count'], nodata=float('nan'),
        add_stats={'class_count': lambda x: float((x[~np.isnan(x)] == value).sum())}
    )
    return [s['class_count'] for s in stats]

wards_water = wards.copy()
wards_water['permanent']      = count_class(tmp_path, wards, 1)
wards_water['lost_permanent'] = count_class(tmp_path, wards, 3)
wards_water['seasonal']       = count_class(tmp_path, wards, 4)
wards_water['lost_seasonal']  = count_class(tmp_path, wards, 6)
wards_water['total_pixels']   = (wards_water['permanent'] + wards_water['lost_permanent'] +
                                  wards_water['seasonal']  + wards_water['lost_seasonal'])
wards_water['water_loss_pct'] = (
    (wards_water['lost_permanent'] + wards_water['lost_seasonal']) /
    wards_water['total_pixels'].replace(0, np.nan) * 100
).fillna(0).round(1)

# Build scatter point coordinates clipped to county boundary
def raster_to_points(arr, values):
    rows, cols = np.where(np.isin(arr, values))
    lons = [GRID_TRANS.c + (c + 0.5) * GRID_TRANS.a for c in cols]
    lats = [GRID_TRANS.f + (r + 0.5) * GRID_TRANS.e for r in rows]
    return filter_to_county(lons, lats)

print('Building scatter points (clipped to county boundary)...')
lons_p, lats_p = raster_to_points(water_transition, [1, 4])   # present
lons_l, lats_l = raster_to_points(water_transition, [3, 6])   # lost
print(f'Present water points inside county: {len(lons_p)}')
print(f'Lost water points inside county:    {len(lons_l)}')

fig, axes = plt.subplots(1, 2, figsize=(20, 14))

# Panel A: present vs lost water sources
ax = axes[0]
wards.plot(ax=ax, color='#F0EDE8', edgecolor='#AAAAAA', linewidth=0.4, zorder=1)
if lons_p:
    ax.scatter(lons_p, lats_p, c='#2E75B6', s=35, zorder=3,
               marker='s', linewidths=0, label=f'Water still present (n={len(lons_p)})')
if lons_l:
    ax.scatter(lons_l, lats_l, c='#C00000', s=35, zorder=3,
               marker='s', linewidths=0, label=f'Water lost since 1984 (n={len(lons_l)})')
wards.boundary.plot(ax=ax, color='#555555', linewidth=0.5, zorder=4)
ax.set_xlim(BOUNDS['west'], BOUNDS['east'])
ax.set_ylim(BOUNDS['south'], BOUNDS['north'])
ax.legend(fontsize=9, loc='lower left')
ax.set_title('Surface Water Sources — Present and Lost\n'
             'JRC Global Surface Water | 1984 to 2021',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
add_north_arrow(ax)
add_caption(ax, 'Blue = water sources still present. Red = sources that existed in 1984 but are now gone.')

# Panel B: ward-level water loss choropleth
ax = axes[1]
wards_water.plot(
    column='water_loss_pct', cmap='Reds', linewidth=0.5, edgecolor='white',
    legend=True, vmin=0, vmax=100,
    legend_kwds={'label': '% of ward water sources lost since 1984',
                 'orientation': 'vertical'},
    ax=ax
)
ax.set_xlim(BOUNDS['west'], BOUNDS['east'])
ax.set_ylim(BOUNDS['south'], BOUNDS['north'])
ax.set_title('Ward-Level Water Source Loss — Kitui County\n'
             '% of water pixels that existed in 1984 but are no longer detected',
             fontsize=11, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
add_north_arrow(ax)
add_caption(ax, 'Darker red = higher proportion of water sources lost since 1984. '
               'White = no historical water detected in this ward.')

plt.suptitle('Seasonal Water Availability and Water Source Loss — Kitui County, Kenya\n'
             'JRC Global Surface Water Analysis 1984 to 2021',
             fontsize=13, fontweight='bold')
plt.tight_layout()
path = MAPS + 'fig01_seasonal_water_availability.png'
plt.savefig(path)
plt.show()
print(f'Figure 1 saved: {path}')


### 4. Figure 2 — Vegetation Stress and Land Condition Map

**Phase 1 deliverable:** Shows vegetation condition and stress across Kitui
using 25 years of MODIS satellite data (2000 to 2025).

Three panels:
- **Current NDVI** — mean vegetation greenness 2000 to 2025
- **Baseline NDVI** — mean vegetation greenness 2000 to 2004 (earliest available window)
- **Vegetation anomaly** — decline from baseline (positive = worse than baseline)

Areas with significant decline indicate land degradation and drought stress.
These areas typically have lower groundwater recharge and reduced water retention capacity.


In [ ]:
# ── Figure 2: Vegetation stress and land condition ────────────────────────────
print('Loading NDVI layers from GEE Assets...')
ndvi_current  = mask(load_asset('kitui_ndvi_mean_2000_2025',     download_scale=500))
ndvi_baseline = mask(load_asset('kitui_ndvi_baseline_2000_2004', download_scale=500))
print('Loaded')

ndvi_anomaly = np.where(
    (~np.isnan(ndvi_current)) & (~np.isnan(ndvi_baseline)),
    ndvi_baseline - ndvi_current,
    np.nan
)

ext = [BOUNDS['west'], BOUNDS['east'], BOUNDS['south'], BOUNDS['north']]
fig, axes = plt.subplots(1, 3, figsize=(24, 14))

# Panel A: current NDVI
ax = axes[0]
valid = ndvi_current[~np.isnan(ndvi_current)]
im = ax.imshow(ndvi_current, cmap='RdYlGn', vmin=0.1, vmax=0.8,
               extent=ext, origin='upper', aspect='equal')
wards.boundary.plot(ax=ax, color='#333333', linewidth=0.4)
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04,
             label='NDVI (0 = bare soil, 1 = dense vegetation)')
ax.set_title('Current Vegetation Condition\nMean NDVI 2000 to 2025 (MODIS MOD13A3)',
             fontsize=10, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
add_north_arrow(ax)
add_caption(ax, f'County mean NDVI: {valid.mean():.3f}')

# Panel B: baseline NDVI
ax = axes[1]
valid_b = ndvi_baseline[~np.isnan(ndvi_baseline)]
im2 = ax.imshow(ndvi_baseline, cmap='RdYlGn', vmin=0.1, vmax=0.8,
                extent=ext, origin='upper', aspect='equal')
wards.boundary.plot(ax=ax, color='#333333', linewidth=0.4)
plt.colorbar(im2, ax=ax, fraction=0.046, pad=0.04,
             label='NDVI (0 = bare soil, 1 = dense vegetation)')
ax.set_title('Baseline Vegetation Condition\nMean NDVI 2000 to 2004 (earliest MODIS window)',
             fontsize=10, fontweight='bold')
ax.set_xlabel('Longitude')
add_north_arrow(ax)
add_caption(ax, 'MODIS begins in 2000. Pre-2000 baseline data is not available.')

# Panel C: anomaly
ax = axes[2]
valid_a = ndvi_anomaly[~np.isnan(ndvi_anomaly)]
vmax_a = np.percentile(np.abs(valid_a), 98)
im3 = ax.imshow(ndvi_anomaly, cmap='RdYlGn_r', vmin=-vmax_a, vmax=vmax_a,
                extent=ext, origin='upper', aspect='equal')
wards.boundary.plot(ax=ax, color='#333333', linewidth=0.4)
cbar3 = plt.colorbar(im3, ax=ax, fraction=0.046, pad=0.04)
cbar3.set_label('NDVI change (positive = decline below baseline)')
ax.set_title('Vegetation Stress — Decline from Baseline\n'
             'Positive = vegetation worse than 2000 to 2004 baseline',
             fontsize=10, fontweight='bold')
ax.set_xlabel('Longitude')
add_north_arrow(ax)
pct_decline = (valid_a > 0).mean() * 100
add_caption(ax, f'{pct_decline:.1f}% of county shows vegetation below baseline. '
                'Red = degraded. Green = improved.')

plt.suptitle('Vegetation Stress and Land Condition — Kitui County, Kenya\n'
             '25 Years of MODIS Satellite Data (2000 to 2025)',
             fontsize=14, fontweight='bold')
plt.tight_layout()
path = MAPS + 'fig02_vegetation_stress.png'
plt.savefig(path)
plt.show()
print(f'Figure 2 saved: {path}')


### 5. Figure 3 — WASI Ward Choropleth

Publication-ready ward-level WASI map for the Phase 1 report.
Labels the 10 highest-stress wards.


In [ ]:
# ── Figure 3: WASI ward choropleth ────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 14))
wasi_wards.plot(
    column='WASI_mean', cmap='RdYlGn_r', linewidth=0.6, edgecolor='white',
    legend=True, vmin=0, vmax=1,
    legend_kwds={'label': 'Water Access Stress Index (0 = low stress, 1 = high stress)',
                 'orientation': 'vertical', 'shrink': 0.7},
    ax=ax
)
for _, row in wasi_wards.nlargest(10, 'WASI_mean').iterrows():
    c = row.geometry.centroid
    ax.annotate(row['Ward'], xy=(c.x, c.y), fontsize=6.5,
                ha='center', va='center', fontweight='bold',
                bbox=dict(boxstyle='round,pad=0.2', fc='white', alpha=0.6, ec='none'))
ax.set_title('Water Access Stress Index — Kitui County\n'
             'Phase 1 satellite analysis: 40 wards, 500m resolution',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
add_north_arrow(ax)
add_caption(ax, 'C1 (distance to boreholes) not included — borehole dataset pending. '
                'Source: CHIRPS, MODIS, WorldPop, SRTM via Google Earth Engine.')
plt.tight_layout()
path = MAPS + 'fig03_wasi_choropleth.png'
plt.savefig(path)
plt.show()
print(f'Figure 3 saved: {path}')


### 6. Figure 4 — WASI Component Breakdown Chart

Horizontal stacked bar chart showing the contribution of each component
to the WASI score for every ward, ordered from lowest to highest stress.
Shows which stress driver dominates in each ward.


In [ ]:
# ── Figure 4: WASI component breakdown chart ──────────────────────────────────
wasi_sorted  = wasi_table.sort_values('WASI_mean', ascending=True).copy()
comp_cols    = ['C2_Rainfall', 'C3_NDVI', 'C4_Population', 'C5_Slope']
comp_labels  = ['C2: Rainfall variability (35.7%)', 'C3: NDVI anomaly (21.4%)',
                'C4: Population (28.6%)', 'C5: Slope (14.3%)']
comp_weights = [0.357, 0.214, 0.286, 0.143]
comp_colors  = ['#2E75B6', '#70AD47', '#ED7D31', '#FFC000']

fig, ax = plt.subplots(figsize=(14, 12))
y_pos = np.arange(len(wasi_sorted))
cumulative = np.zeros(len(wasi_sorted))
for col, label, w, colour in zip(comp_cols, comp_labels, comp_weights, comp_colors):
    if col in wasi_sorted.columns:
        vals = wasi_sorted[col].fillna(0).values * w
        ax.barh(y_pos, vals, left=cumulative, height=0.7,
                color=colour, alpha=0.85, label=label)
        cumulative += vals
ax.scatter(wasi_sorted['WASI_mean'].values, y_pos,
           color='black', s=20, zorder=5, label='WASI composite')
ax.set_yticks(y_pos)
ax.set_yticklabels(wasi_sorted['Ward'], fontsize=7.5)
ax.set_xlabel('Weighted stress contribution (0 = no stress, 1 = maximum stress)')
ax.set_title('WASI Component Breakdown by Ward — Kitui County\n'
             'Phase 1: four satellite components, C1 (boreholes) pending',
             fontsize=12, fontweight='bold')
ax.legend(loc='lower right', fontsize=8)
ax.axvline(0.55, color='red', linestyle='--', alpha=0.4, linewidth=1)
ax.text(0.555, len(wasi_sorted) - 1, 'High stress\nthreshold',
        fontsize=7, color='red', alpha=0.6)
ax.set_xlim(0, 0.75)
ax.grid(axis='x', alpha=0.3)
plt.tight_layout()
path = MAPS + 'fig04_wasi_components.png'
plt.savefig(path)
plt.show()
print(f'Figure 4 saved: {path}')


### 7. Figure 5 — Hotspot Analysis Map

Ward-level hotspot classification from Notebook 03.
Shows statistically significant clusters of high and low water stress.


In [ ]:
# ── Figure 5: Hotspot analysis map ────────────────────────────────────────────
import shutil

src = OUT + 'kitui_hotspot_map.png'
if os.path.exists(src):
    shutil.copy(src, MAPS + 'fig05_hotspot_map.png')
    print('Notebook 03 hotspot map copied to report figures folder')

WARD_COLOURS = {
    'Hotspot':         '#C00000',
    'Not significant': '#D9D9D9',
    'Coldspot':        '#2E75B6',
    'No data':         '#F0F0F0',
}
fig, ax = plt.subplots(figsize=(12, 14))
ward_class_col = 'Ward_Class' if 'Ward_Class' in hotspot_wards.columns else None
if ward_class_col:
    for cls, colour in WARD_COLOURS.items():
        subset = hotspot_wards[hotspot_wards[ward_class_col] == cls]
        if len(subset) > 0:
            subset.plot(ax=ax, color=colour, linewidth=0.5, edgecolor='white')
    for _, row in hotspot_wards[hotspot_wards[ward_class_col] == 'Hotspot'].iterrows():
        c = row.geometry.centroid
        ax.annotate(row['Ward'], xy=(c.x, c.y), fontsize=7,
                    ha='center', va='center', fontweight='bold', color='white',
                    bbox=dict(boxstyle='round,pad=0.15', fc='#C00000', alpha=0.7, ec='none'))
else:
    hotspot_wards.plot(ax=ax, color='#D9D9D9', linewidth=0.5, edgecolor='white')

legend_patches = [
    mpatches.Patch(color='#C00000', label='Hotspot — significant high-stress cluster'),
    mpatches.Patch(color='#D9D9D9', label='Not significant'),
    mpatches.Patch(color='#2E75B6', label='Coldspot — significant low-stress cluster'),
]
ax.legend(handles=legend_patches, loc='lower left', fontsize=8)
ax.set_title("Water Stress Spatial Clustering — Kitui County\n"
             "Getis-Ord Gi* | 500m raster | Moran's I = 0.332 (p=0.001)",
             fontsize=12, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
add_north_arrow(ax)
add_caption(ax, 'Ward classified as hotspot if >=30% of pixels have Gi* z-score > 1.96.')
plt.tight_layout()
path = MAPS + 'fig05_hotspot_ward.png'
plt.savefig(path)
plt.show()
print(f'Figure 5 saved: {path}')


### 8. Phase 2 Figures

Figures 6 and 7 require borehole data and outputs from Notebooks 04 and 05.
This cell runs automatically if those outputs are present on Drive
and skips them if they are not.

Once borehole data is received and Notebooks 04 and 05 have been run,
re-run this cell to generate the remaining figures.


In [ ]:
# ── Phase 2 figures ────────────────────────────────────────────────────────────
if has_boreholes and has_coverage:
    print('Phase 2 outputs detected — generating Figures 6 and 7...')

    fig, ax = plt.subplots(figsize=(12, 14))
    wards.plot(ax=ax, color='#F5F5F5', edgecolor='#CCCCCC', linewidth=0.5)
    if 'Functional' in boreholes.columns:
        functional_bh    = boreholes[boreholes['Functional'] == True]
        nonfunctional_bh = boreholes[boreholes['Functional'] == False]
        functional_bh.plot(ax=ax, color='#2E75B6', markersize=5, alpha=0.7,
                           label=f'Functional ({len(functional_bh)})')
        if len(nonfunctional_bh) > 0:
            nonfunctional_bh.plot(ax=ax, color='#C00000', markersize=4,
                                  alpha=0.6, marker='x',
                                  label=f'Non-functional ({len(nonfunctional_bh)})')
    else:
        boreholes.plot(ax=ax, color='#2E75B6', markersize=5, alpha=0.7)
    ax.legend(fontsize=9, loc='lower left')
    ax.set_title('Borehole Network — Kitui County', fontsize=12, fontweight='bold')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    add_north_arrow(ax)
    plt.tight_layout()
    plt.savefig(MAPS + 'fig06_borehole_network.png')
    plt.show()
    print('Figure 6 saved')

    fig, ax = plt.subplots(figsize=(12, 14))
    coverage_wards.plot(
        column='Pop_Gap_Pct_1km', cmap='Reds', linewidth=0.5, edgecolor='white',
        legend=True, vmin=0, vmax=100,
        legend_kwds={'label': '% population outside 1km borehole coverage',
                     'orientation': 'vertical'},
        ax=ax
    )
    ax.set_title('Population Coverage Gap — Kitui County\n1km walking threshold',
                 fontsize=12, fontweight='bold')
    ax.set_xlabel('Longitude')
    ax.set_ylabel('Latitude')
    add_north_arrow(ax)
    plt.tight_layout()
    plt.savefig(MAPS + 'fig07_coverage_gap.png')
    plt.show()
    print('Figure 7 saved')
else:
    print('Phase 2 outputs not yet available.')
    print('Run Notebooks 04 and 05 after borehole data is received, then re-run this cell.')


### 9. Export Report Tables and Summary

Saves clean CSV versions of the key analysis tables for the report.
Prints a complete list of all figures produced.


In [ ]:
# ── Export report tables and summary ──────────────────────────────────────────
t1 = wasi_table.sort_values('WASI_mean', ascending=False).copy()
t1.to_csv(REPT + 'table01_wasi_ward.csv', index=False)
print(f'Table 1 saved: table01_wasi_ward.csv ({len(t1)} wards)')

if has_coverage:
    coverage_table.to_csv(REPT + 'table02_coverage_gap.csv', index=False)
    print(f'Table 2 saved: table02_coverage_gap.csv')
else:
    print('Table 2 (coverage gap): not yet available — run Notebook 04')

print()
print('REPORT FIGURES COMPLETE')
print('=' * 50)
print(f'Saved to: {MAPS}')
print()
phase1_figs = sorted([f for f in os.listdir(MAPS) if f.endswith('.png')])
for f in phase1_figs:
    print(f'  {f}')
print()
print('Phase 1 deliverables:')
print('  fig01 — Seasonal water availability and water source loss map')
print('  fig02 — Vegetation stress and land condition map')
print('  fig03 — Water Access Stress Index map')
print('  fig04 — WASI component breakdown chart')
print('  fig05 — Spatial hotspot analysis map')
print()
print('Phase 2 figures will be generated once borehole data is received.')
